validate data quality


In [0]:

# %pip install databricks-labs-dqx
!pip install databricks-labs-dqx  



In [0]:
dbutils.library.restartPython()

In [0]:
raw_post_df=spark.read.table("default.raw_posts")

In [0]:
from databricks.labs.dqx import check_funcs
from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx.rule import DQRowRule, DQDatasetRule, DQForEachColRule
from databricks.labs.dqx.config import InputConfig, OutputConfig
from databricks.sdk import WorkspaceClient

dq_engine=DQEngine(WorkspaceClient())

checks=[
    # "Id","CreationDate" Must NOT be NULL
    *DQForEachColRule(
        columns= ["Id","CreationDate"],
        criticality="error",
        check_func=check_funcs.is_not_null
    ).get_rules(),
    # CreationDate must not be greater than today.
    DQRowRule(
        name="creation_data_not_in_future",
        criticality="error",
        check_func=check_funcs.is_not_in_future,
        column="CreationDate"   
    ),
    DQRowRule(
        name="post_type_id_allowed_values",
        criticality="warn",
        check_func=check_funcs.is_in_list,
        column="PostTypeId",
        check_func_kwargs={"allowed":["1","2","3","4"]}
    )
]

valid_df,quarantined_df=dq_engine.apply_checks_and_split(df=raw_post_df,checks=checks)

In [0]:
display(quarantined_df.limit(10))

In [0]:
quarantined_df.count()

In [0]:
valid_df.count()

In [0]:
valid_df.write.mode("append").saveAsTable("bronze_posts_table")